In [ ]:
!pip install -U langchain-text-splitters
!pip install -U langchain-community langchain-text-splitters langchain
!pip install pypdf
!pip install -U langchain-huggingface
!pip install -qU langchain-chroma
!pip install -qU langchain
!pip install -qU langchain-groq
!pip install sentence-transformers
!pip install -q gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 9.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1

In [54]:
from langchain_groq import ChatGroq
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
import shutil
import gradio as gr
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
import re
import gc

In [56]:
from google.colab import drive
from google.colab import userdata

# --- MONTAGE DRIVE ET CHEMINS ---
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/Smart-HR-Assistant"
NEW_DATA_PATH = os.path.join(PROJECT_PATH, "nouveau_data")
ACTUEL_DATA_PATH = os.path.join(PROJECT_PATH, "actuel_data")
CHROMA_PATH = os.path.join(PROJECT_PATH, "chroma_db")

# Création des dossiers si nécessaire
for path in [NEW_DATA_PATH, ACTUEL_DATA_PATH, CHROMA_PATH]:
    os.makedirs(path, exist_ok=True)

# --- CONFIGURATION API ---
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
llm_chat = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.5
)

llm_multi_query = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [57]:
# Formuler la question
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", (
    "Ton unique job est de reformuler la question de l'utilisateur pour qu'elle soit "
    "compréhensible toute seule, en utilisant l'historique. "
    "REGLÈS : "
    "1. NE RÉPONDS PAS à la question. "
    "2. NE DONNE AUCUNE INFORMATION sur l'entreprise. "
    "3. Renvoie UNIQUEMENT la question reformulée. "
    "Exemple : Si l'historique parle de Mutuelle et que l'utilisateur dit 'C'est combien ?', "
    "tu réponds juste : 'Quel est le prix de la mutuelle ?'"
    )),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
contextualize_q_chain = contextualize_q_prompt | llm_chat | StrOutputParser()

# Le rédacteur
instruction_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Tu es un expert en analyse de documents RH. Ton rôle est de fournir des réponses "
        "précises et structurées à partir du contexte fourni ci-dessous.\n\n"
        "CONTEXTE :\n{context}\n\n"
        "DIRECTIVES DE PRÉCISION :\n"
        "1. SYNTHÈSE : Synthétise les points clés (noms, dates, montants).\n"
        "2. SOURCES : Pour chaque information importante, tu DOIS citer le document source. "
        "Exemple : 'La prime est de 300€ [Source: politique_primes.pdf]'.\n" # <--- LA CLÉ !
        "3. INTÉGRITÉ : Si l'info n'est pas dans le contexte, dis 'Je ne trouve pas cette info dans les documents fournis'.\n"
        "4. ANTI-MÉLANGE : Si deux documents se contredisent, précise-le (ex: 'Le doc A dit X, mais le doc B dit Y').\n"
        "5. FORMAT : Utilise des listes à puces et du gras pour les chiffres."
    )),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

In [58]:
def get_or_update_db(update=True):
    os.makedirs(os.path.dirname(CHROMA_PATH), exist_ok=True)
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    # 1. Charger la base existante (ou en créer une vide)
    vector_db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embeddings)

    # 2. Vérifier s'il y a du nouveau
    if update:
      new_files = [f for f in os.listdir(NEW_DATA_PATH) if f.endswith('.pdf')]

      if new_files:
          loader = PyPDFDirectoryLoader(NEW_DATA_PATH)
          new_docs = loader.load()

          text_splitter = RecursiveCharacterTextSplitter(
              chunk_size=1200,
              chunk_overlap=200,
              length_function=len,
              add_start_index=True
          )

          new_chunks = text_splitter.split_documents(new_docs)

          # Ajout à Chroma
          vector_db.add_documents(new_chunks)

          # Déplacement vers l'archive
          for file_name in new_files:
              shutil.move(os.path.join(NEW_DATA_PATH, file_name),
                          os.path.join(ACTUEL_DATA_PATH, file_name))
              print(f"Archivé : {file_name}")
      else:
          print("Aucun nouveau fichier. Utilisation de la base existante.")

    return vector_db

In [59]:
def vider_db():
    # Vider la collection
    try:
        vector_db = get_or_update_db(False)
        vector_db.delete_collection()
        print("Collection vidée.")

    except Exception as e:
        print(f"Erreur lors du vidage : {e}")

    # Supprimer la collection
    for nom in os.listdir(CHROMA_PATH):
        item_path = os.path.join(path, nom)
        if os.path.isdir(item_path):
            try:
                shutil.rmtree(item_path)
                print(f"Dossier de collection supprimé : {nom}")
            except Exception as e:
                print(f"Impossible de supprimer {nom} : {e}")

    # Supprimer les fichiers PDF dans actuel_data
    for nom in os.listdir(ACTUEL_DATA_PATH):
        chemin_complet = os.path.join(ACTUEL_DATA_PATH, nom)
        try:
            if os.path.isfile(chemin_complet):
                os.unlink(chemin_complet)
            elif os.path.isdir(chemin_complet):
                shutil.rmtree(chemin_complet)
            print("Actuel data vidé")
        except Exception as e:
            print(f"Impossible de supprimer {nom} : {e}")

In [43]:
"""# 1. On récupère les données actuelles
data = get_or_update_db(False).get()

# 2. On compte les IDs
nombre_de_morceaux = len(data['ids'])

print(f"🔍 Vérification : La base contient {nombre_de_morceaux} morceaux.")

if nombre_de_morceaux == 0:
    print("✨ Confirmation : La base est totalement vide !")
else:
    print("⚠️ Attention : Il reste encore des données.")"""

'# 1. On récupère les données actuelles\ndata = get_or_update_db(False).get()\n\n# 2. On compte les IDs\nnombre_de_morceaux = len(data[\'ids\'])\n\nprint(f"🔍 Vérification : La base contient {nombre_de_morceaux} morceaux.")\n\nif nombre_de_morceaux == 0:\n    print("✨ Confirmation : La base est totalement vide !")\nelse:\n    print("⚠️ Attention : Il reste encore des données.")'

In [44]:
# Exemple de recherche avec sources
"""vector_db = get_or_update_db()
query = "c'est quoi le regle interieur?"
docs = vector_db.similarity_search(query, k=3)

print("--- ANALYSE DES DOUBLONS ---")
for i, doc in enumerate(docs):
    # On affiche l'index de départ pour voir si c'est le même endroit
    index = doc.metadata.get('start_index', 'N/A')
    print(f"Morceau {i+1} | Source: {os.path.basename(doc.metadata['source'])}")
    print(f"Position dans le doc: {index}")
    print(f"Texte: {doc.page_content[:100].strip()}...")
    print("-" * 30)"""

'vector_db = get_or_update_db()\nquery = "c\'est quoi le regle interieur?"\ndocs = vector_db.similarity_search(query, k=3)\n\nprint("--- ANALYSE DES DOUBLONS ---")\nfor i, doc in enumerate(docs):\n    # On affiche l\'index de départ pour voir si c\'est le même endroit\n    index = doc.metadata.get(\'start_index\', \'N/A\')\n    print(f"Morceau {i+1} | Source: {os.path.basename(doc.metadata[\'source\'])}")\n    print(f"Position dans le doc: {index}")\n    print(f"Texte: {doc.page_content[:100].strip()}...")\n    print("-" * 30)'

In [78]:
def respond(message, chat_history):
    global llm_chat, llm_multi_query, contextualize_q_chain

    # 1. Mise à jour de la base (ou récupération)
    vector_db = get_or_update_db()

    # --- NOUVEAU : TRANSFORME L'HISTORIQUE POUR LE LLM ---
    formatted_history = []
    for user_msg, ai_msg in chat_history:
        formatted_history.append(HumanMessage(content=user_msg))
        formatted_history.append(AIMessage(content=ai_msg))

    # --- ÉTAPE 2 : REFORMULATION (Le Secrétaire travaille ici) ---
    # Si c'est la 1ère question, on garde le message tel quel.
    # Sinon, on demande au LLM de clarifier la question par rapport à l'historique.
    if chat_history:
        standalone_question = contextualize_q_chain.invoke({
            "chat_history": formatted_history,
            "input": message
        })
        print(f"DEBUG - Question reformulée : {standalone_question} okk")
    else:
        standalone_question = message

    # --- ÉTAPE 3 : RECHERCHE AVANCÉE (Multi-Query) ---
    # On crée le retriever intelligent qui va générer des questions synonymes
    advanced_retriever = MultiQueryRetriever.from_llm(
        retriever=vector_db.as_retriever(search_kwargs={"k": 3}),
        llm=llm_multi_query
    )

    docs = advanced_retriever.invoke(input=standalone_question)
    context = "\n\n".join([d.page_content for d in docs])

    # --- ÉTAPE 4 : RÉPONSE FINALE (Le Rédacteur) ---
    chain = instruction_prompt | llm_chat
    result = chain.invoke({
        "context": context,
        "question": message, # On garde la question originale pour l'affichage
        "chat_history": formatted_history
    })

    # 5. Ajouter l'échange à l'historique
    chat_history.append((message, result.content))
    return "", chat_history

In [77]:
respond("c'est quoi la regle ?", [])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Aucun nouveau fichier. Utilisation de la base existante.

--- TEST DAY 6 ---
Nombre de morceaux uniques trouvés : 10
Morceau 1 : professionnel.   Le télétravailleur s’engage à rés...
Morceau 2 : Un référent du télétravail sera nommé afin de répo...
Morceau 3 : Le télétravailleur ne reçoit pas de public et ne f...
Morceau 4 : 2    SOMMAIRE    Préambule             p3    Cadre...
Morceau 5 : conditions optimales.  Chapitre 2 - Modalités de m...
Morceau 6 : identifier :  Les travaux rédactionnels comme les ...
Morceau 7 : Les jours de télétravail sont, en principe, fixes ...
Morceau 8 : 1               Charte du         télétravail     ...
Morceau 9 : 9    Dans la limite de 30 jours par an pour un tem...
Morceau 10 : ► Disponibilité, flexibilité intellectuelle et cur...
------------------



('',
 [("c'est quoi la regle ?",
   "Je n'ai trouvé aucune règle spécifique mentionnée dans le texte fourni. Cependant, je peux vous aider à identifier les règles et les informations clés liées au télétravail dans ce contexte.\n\nVoici quelques règles et informations clés que j'ai identifiées :\n\n**Règles de télétravail**\n\n* Le télétravailleur s'engage à réserver l'exclusivité de son travail à sa hiérarchie et à veiller à ce que les informations sensibles traitées à domicile demeurent confidentielles.\n* Les dossiers ou documents papier s'originaux ainsi que les documents partagés doivent rester dans les locaux de la structure.\n* Les jours de télétravail sont, en principe, fixes pour la plupart des postes mais peuvent être flexibles pour s'adapter à l'activité.\n* Les jours de télétravail ne peuvent pas être accolés ou encadrés un week-end ou un jour férié.\n* En cas d'impossibilité de télétravailler le jour prévu, l'agent doit se rendre sur son lieu de travail.\n\n**Conditions d'é

In [79]:
import shutil

def upload_files(files):
    # On s'assure que le dossier existe
    if not os.path.exists(NEW_DATA_PATH):
        os.makedirs(NEW_DATA_PATH)

    file_paths = []
    for file in files:
        # file.name contient le chemin temporaire du fichier téléchargé
        nom_fichier = os.path.basename(file.name)
        destination = os.path.join(NEW_DATA_PATH, nom_fichier)

        # On copie le fichier du dossier temporaire vers Google Drive
        shutil.copy(file.name, destination)
        file_paths.append(destination)
    return f"SUCCÈS : {len(file_paths)} fichiers ajoutés. Cliquez maintenant sur Synchroniser."

In [80]:
vider_db()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection vidée.
Dossier de collection supprimé : 6540824d-6cca-499b-834e-c0fdcf898700
Actuel data vidé
Actuel data vidé


In [81]:
with gr.Blocks() as demo:
    gr.Markdown("# 📁 Assistant Smart-HR - Espace Administration")

    with gr.Row():
        # Partie Upload pour le RH
        with gr.Column(scale=1):
            file_output = gr.File(file_count="multiple", label="Téléverser des PDF RH")
            upload_button = gr.Button("1. Stocker les fichiers")
            status_label = gr.Textbox(label="Statut du stockage")

            update_db_button = gr.Button("2. Synchroniser la base (IA)", variant="primary")
            db_status = gr.Textbox(label="Statut de l'IA")

        # Partie Chat pour l'utilisateur
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Discussion avec vos documents")
            msg = gr.Textbox(label="Posez votre question RH")
            clear = gr.Button("Effacer l'historique")

    # --- LOGIQUE DES BOUTONS ---

    # Bouton 1 : On stocke les fichiers dans le Drive
    upload_button.click(upload_files, inputs=file_output, outputs=status_label, show_progress="hidden", queue=False)

    # Bouton 2 : On lance la fonction maj_base_de_donnees() qu'on a fait juste avant
    update_db_button.click(get_or_update_db, outputs=db_status)

    # Le reste de ta logique de chat (respond...)
    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

demo.launch(debug=True)

/tmp/ipython-input-13885/3760806977.py:16: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Discussion avec vos documents")
/tmp/ipython-input-13885/3760806977.py:16: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Discussion avec vos documents")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fc11a8fcd8a9732c87.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fc11a8fcd8a9732c87.gradio.live
